# Team Summary Report

A supporter-focused summary of one team's performance in a Rugby Tracker competition.

Set `COMPETITION` and `TEAM` in the next cell, then run all cells. Names are matched case-insensitively. If a competition name occurs in more than one season, set `SEASON` too.

In [ ]:
COMPETITION = ""
TEAM = ""

# Example: "2026"; leave as None when the competition name is unique
SEASON = None  

## Setup and data loading

In [ ]:
from __future__ import annotations

import os
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_rows", 100)
plt.style.use("seaborn-v0_8-whitegrid")

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "rugby_tracker").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from within the RugbyTracker repository.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DB_PATH = Path(os.environ.get("RUGBY_TRACKER_DB", REPO_ROOT / "data" / "rugbytracker.db")).expanduser()
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Rugby Tracker database not found: {DB_PATH}")

sys.path.insert(0, str(REPO_ROOT / "src"))
from rugby_tracker.standings import calculate_competition  # noqa: E402

connection = sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
print(f"Using database: {DB_PATH}")

In [ ]:
competition_sql = "SELECT id, name, season, gender, ruleset FROM competitions WHERE name = ? COLLATE NOCASE"
params = [COMPETITION.strip()]
if SEASON is not None:
    competition_sql += " AND season = ? COLLATE NOCASE"
    params.append(str(SEASON).strip())
competitions = [dict(row) for row in connection.execute(competition_sql, params)]
if not competitions:
    available = pd.read_sql_query("SELECT name, season FROM competitions ORDER BY name, season", connection)
    raise ValueError(f"Competition not found: {COMPETITION!r} (season={SEASON!r}). Available:\n{available.to_string(index=False)}")
if len(competitions) > 1:
    seasons = ", ".join(str(row["season"]) for row in competitions)
    raise ValueError(f"Competition {COMPETITION!r} exists in multiple seasons ({seasons}); set SEASON.")
competition = competitions[0]
if not competition["ruleset"]:
    raise ValueError(f"{competition['name']} {competition['season']} has no standings ruleset.")

matches_sql = """
SELECT m.*, h.name AS home_team_name, hc.name AS home_team_country,
       a.name AS away_team_name, ac.name AS away_team_country,
       v.name AS venue_name
FROM matches m
JOIN teams h ON h.id = m.home_team_id
JOIN countries hc ON hc.id = h.country_id
JOIN teams a ON a.id = m.away_team_id
JOIN countries ac ON ac.id = a.country_id
LEFT JOIN venues v ON v.id = m.venue_id
WHERE m.competition_id = ?
ORDER BY m.match_date, COALESCE(m.kickoff_time, ''), m.id
"""
all_matches = [dict(row) for row in connection.execute(matches_sql, (competition["id"],))]
team_names = sorted({m["home_team_name"] for m in all_matches} | {m["away_team_name"] for m in all_matches}, key=str.casefold)
matched_names = [name for name in team_names if name.casefold() == TEAM.strip().casefold()]
if not matched_names:
    raise ValueError(f"Team not found in {competition['name']} {competition['season']}: {TEAM!r}. Available: {', '.join(team_names)}")
team_name = matched_names[0]
team_row = connection.execute("""
    SELECT t.id, t.name, c.name AS country, v.name AS home_venue
    FROM teams t JOIN countries c ON c.id = t.country_id
    JOIN venues v ON v.id = t.home_venue_id
    WHERE t.name = ? COLLATE NOCASE
""", (team_name,)).fetchone()
team = dict(team_row)
completed_matches = [m for m in all_matches if m["home_score"] is not None and m["away_score"] is not None]
team_matches = [m for m in completed_matches if team_name in (m["home_team_name"], m["away_team_name"])]
if not team_matches:
    raise ValueError(f"No completed matches found for {team_name} in this competition.")

In [ ]:
def team_view(match: dict) -> dict:
    is_home = match["home_team_name"] == team_name
    scored = match["home_score"] if is_home else match["away_score"]
    conceded = match["away_score"] if is_home else match["home_score"]
    tries_for = match["home_tries"] if is_home else match["away_tries"]
    tries_against = match["away_tries"] if is_home else match["home_tries"]
    result = "W" if scored > conceded else "L" if scored < conceded else "D"
    return {
        "Round": match["round"] or "—", "Date": pd.to_datetime(match["match_date"]).date(),
        "Opponent": match["away_team_name"] if is_home else match["home_team_name"],
        "Home/Away": "Home" if is_home else "Away", "Score": f"{scored}–{conceded}",
        "Result": result, "Points scored": scored, "Points conceded": conceded,
        "Tries scored": tries_for, "Tries conceded": tries_against,
        "Margin": scored - conceded, "Total points": scored + conceded,
    }

results = pd.DataFrame(team_view(match) for match in team_matches)
played = len(results)
won = int((results["Result"] == "W").sum())
drawn = int((results["Result"] == "D").sum())
lost = int((results["Result"] == "L").sum())
points_for = int(results["Points scored"].sum())
points_against = int(results["Points conceded"].sum())
tries_for = int(results["Tries scored"].sum())
tries_against = int(results["Tries conceded"].sum())
competition_result = calculate_competition(all_matches, competition["ruleset"])
standing = next(row for row in competition_result["table"] if row["Team"] == team_name)
awards = competition_result["awards"]
champions = awards.get("champion") == team_name

def cards(items):
    boxes = "".join(f"<div style='padding:12px 16px;border:1px solid #ddd;border-radius:8px;min-width:120px'><small>{label}</small><br><b style='font-size:1.25em'>{value}</b></div>" for label, value in items)
    display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap;margin:8px 0 18px'>{boxes}</div>"))

## Team overview

In [ ]:
display(HTML(f"<h2>{team_name}</h2><p><b>{competition['name']} · {competition['season']}</b></p>"))
cards([("Country", team["country"]), ("Home venue", team["home_venue"]), ("Matches played", played)])

## Season record

In [ ]:
cards([("Played", played), ("Won", won), ("Drawn", drawn), ("Lost", lost), ("Win percentage", f"{won / played:.1%}")])

## League performance

In [ ]:
status = "Champions" if champions else ("In progress" if not competition_result["complete"] else "Completed")
cards([("League position", standing["Pos"]), ("Competition points", standing["Pts"]), ("Points difference", f"{standing['PD']:+d}"), ("Status", status)])
if competition_result["validation_errors"]:
    display(HTML("<p><i>Table is provisional: " + "; ".join(competition_result["validation_errors"]) + "</i></p>"))
display(HTML("<p><small>Playoff qualification and relegation are not currently stored by Rugby Tracker, so they are not inferred here.</small></p>"))

## Scoring and try summary

In [ ]:
cards([("Points scored", points_for), ("Points conceded", points_against), ("Points difference", f"{points_for - points_against:+d}"), ("Avg scored / match", f"{points_for / played:.1f}"), ("Avg conceded / match", f"{points_against / played:.1f}")])
cards([("Tries scored", tries_for), ("Tries conceded", tries_against), ("Avg tries scored / match", f"{tries_for / played:.1f}"), ("Avg tries conceded / match", f"{tries_against / played:.1f}")])

## Home and away record

In [ ]:
def split_record(location):
    subset = results[results["Home/Away"] == location]
    return {
        "Venue": location, "P": len(subset), "W": int((subset["Result"] == "W").sum()),
        "D": int((subset["Result"] == "D").sum()), "L": int((subset["Result"] == "L").sum()),
        "Points scored": int(subset["Points scored"].sum()), "Points conceded": int(subset["Points conceded"].sum()),
    }
home_away = pd.DataFrame([split_record("Home"), split_record("Away")]).set_index("Venue")
display(home_away)

## Biggest results

In [ ]:
def describe(row):
    return f"{row['Opponent']} ({row['Home/Away'].lower()}), {row['Score']} — {row['Date']:%d %b %Y}"

wins = results[results["Result"] == "W"]
losses = results[results["Result"] == "L"]
largest_win = wins.loc[wins["Margin"].idxmax()] if not wins.empty else None
largest_loss = losses.loc[losses["Margin"].idxmin()] if not losses.empty else None
big_results = pd.DataFrame([
    ("Largest victory", describe(largest_win) if largest_win is not None else "No victories"),
    ("Largest defeat", describe(largest_loss) if largest_loss is not None else "No defeats"),
    ("Highest-scoring match", describe(results.loc[results["Total points"].idxmax()])),
    ("Lowest-scoring match", describe(results.loc[results["Total points"].idxmin()])),
    ("Biggest winning margin", f"{int(largest_win['Margin'])} points" if largest_win is not None else "—"),
    ("Biggest losing margin", f"{abs(int(largest_loss['Margin']))} points" if largest_loss is not None else "—"),
], columns=["Highlight", "Match"]).set_index("Highlight")
display(big_results)

## Match results

In [ ]:
display(results[["Round", "Date", "Opponent", "Home/Away", "Score", "Result"]].style.hide(axis="index"))

fig, ax = plt.subplots(figsize=(9, 3.8))
round_labels = results["Round"].astype(str).tolist()
x = range(1, played + 1)
ax.plot(x, results["Points scored"], marker="o", linewidth=2, label=team_name)
ax.plot(x, results["Points conceded"], marker="o", linewidth=2, label="Opponent")
ax.set(title="Points scored by match", xlabel="Round", ylabel="Points", xticks=list(x), xticklabels=round_labels)
ax.legend(); plt.tight_layout(); plt.show()

---
**Notes.** Only completed fixtures are included in the season, scoring, try, home/away, biggest-result and match-result sections. League calculations use the competition ruleset stored by Rugby Tracker and may exclude knockout rounds. Tied highlights use the first match chronologically.